![image.jpeg](PlayStore.jpeg)

# **Data Cleaning and Analysis Activities**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

**`Note:` it's better to make a copy from the dataframe and test on it before making any changes in the orignal one**

### 1.Which of the following column(s) has/have null values?

Select the columns that you have identified having null/missing values.

In [ ]:
df = pd.read_csv("googleplaystore.csv")
df

In [ ]:
df.info()


In [ ]:
missingVal = df.isna().sum()
print(missingVal[missingVal > 0])
 # columns with null values and number of null values in each columns

### 2. Clean the Rating Column and Other Columns Containing Null Values

**Steps:**

0. Try plotting a histogram and a boxplot for this column to understand the issue
1. Remove invalid values from the Rating column (set them as NaN).
2. Fill null values in the Rating column using the mean().
3. Clean any other non-numerical columns by dropping the rows containing null values.
4. Perform the modifications "in place", modifying `df`. If you make a mistake, re-load the data.

**Details:**
- Replace all ratings not in the range of 0 to 5 with NaN.
- Drop rows with null values in other columns.

In [ ]:
print(plt.style.available)
plt.style.use('classic')

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['Rating'], bins=30, kde=True)
plt.title("Histogram of Ratings")
plt.show()

In [ ]:

plt.subplot(1, 2, 2)
sns.boxplot(x=df['Rating'])
plt.title("Boxplot of Ratings")
plt.show()

In [ ]:
# clean Ratings column
df.loc[(df['Rating'] < 0) | (df['Rating'] > 5), 'Rating'] = None

In [ ]:
df['Rating'].fillna(df['Rating'].mean(), inplace=True)


In [ ]:
df.dropna(subset=['Type', 'Content Rating', 'Current Ver', 'Android Ver'], inplace=True)


In [ ]:
print(df.isnull().sum())


### 3. Clean the Reviews Column and Make It Numeric

You'll notice that some columns from this dataframe which should be numeric, were parsed as object (string). That's because sometimes the numbers are expressed with M, or k to indicate Mega or kilo.

Clean the Reviews column by transforming the values to the correct numeric representation. 

For example, 5M should be 5000000.

In [ ]:

df['Reviews'] = df['Reviews'].replace({'M': '*1e6', 'k': '*1e3'}, regex=True).map(pd.eval).astype(int)


In [ ]:
print(df['Reviews'].dtype)  
print(df['Reviews'].head()) 

### 4. Count the Number of Duplicated Apps

Count the number of duplicated rows. That is, if the app Twitter appears 2 times, that counts as 2.

In [ ]:
# All Duplicated Apps
duplicateApps = df.duplicated(subset=['App'], keep=False).sum()
print("Number of duplicated apps:", duplicateApps)

In [ ]:
# each app and how many times it duplicates
appCounts = df['App'].value_counts()
print(appCounts[appCounts > 1])


### 5. Drop Duplicated Apps Keeping Only the Ones with the Greatest Number of Reviews

Now that the Reviews column is numeric, we can use it to clean duplicated apps. Drop duplicated apps, keeping just one copy of each, the one with the greatest number of reviews.

Hint: you'll need to sort the dataframe by App and Reviews, and that will change the order of your df.

In [ ]:
df = df.sort_values(by=['App', 'Reviews'], ascending=[True, False])

In [ ]:
df.info()

In [ ]:
df = df.drop_duplicates(subset='App', keep='first')
print(df['App'].value_counts())

### 6. Format the Category Column

Categories are all uppercase and words are separated using underscores. Instead, we want them with capitalized in the first character and the underscores transformed as whitespaces.

Example, the category AUTO_AND_VEHICLES should be transformed to: Auto and vehicles. Also, if you find any other wrong value transform it into an Unknown category.

In [ ]:
df['Category'] = df['Category'].str.replace('_', ' ').str.title()
validCategories = df['Category'].unique()  
df.loc[~df['Category'].isin(validCategories), 'Category'] = 'Unknown Category'

print(df['Category'].unique())

### 7. Clean and Convert the Installs Column to Numeric Type

Clean and transform Installs as a numeric type. Some values in Installs will have a + modifier. Just remove the string and honor the original number (for example +2,500 or 2,500+ should be transformed to the number 2500).

In [ ]:
df['Installs'] = df['Installs'].astype(str)
df['Installs'] = df['Installs'].str.replace('+', '', regex=False)  #remove "+"
df['Installs'] = df['Installs'].str.replace(',', '', regex=False)  # remove ","
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')  #convert to numbers

print(df['Installs'].head(20))

### 8. Clean and Convert the Size Column to Numeric (Representing Bytes)

The Size column is of type object. Some values contain either a M or a k that indicate Kilobytes (1024 bytes) or Megabytes (1024 kb). These values should be transformed to their corresponding value in bytes. For example, 898k will become 919552 (898 * 1024).

Some other values are completely invalid (there's no way to infer the numeric type from them). For these, just replace the value for 0.

Some other rules are related to + modifiers, apply the same rules as the previous task.

In [ ]:
df['Size'] = df['Size'].astype(str)  
df['Size'] = df['Size'].str.replace('+', '', regex=False)  
df['Size'] = df['Size'].str.replace(',', '', regex=False) 

df[['Size_value', 'Size_unit']] = df['Size'].str.extract(r'([\d\.]+)([Mk]?)')
# ([\d\.]+)
# \d  ? Matches any digit (0-9).
# \.  ? Matches the decimal point (to handle values like 1.5M).
# +   ? Matches one or more digits or decimal points (e.g? "1.5" or "1000").
# ? This is stored in the 'Size_value' column.
# 
# ([Mk]?)
# [Mk] ? Matches either 'M' (Megabytes) or 'k' (Kilobytes) if present.
# # (?)   ? Makes the presence of 'M' or 'k' optional (some values might be plain numbers like "100").

df['Size_value'] = pd.to_numeric(df['Size_value'], errors='coerce')

df.loc[df['Size_unit'] == 'M', 'Size_value'] *= 1024 * 1024  # 1M = 1024 KB * 1024 B
df.loc[df['Size_unit'] == 'k', 'Size_value'] *= 1024  # 1k = 1024 B

df['Size_value'].fillna(0, inplace=True)


df['Size'] = df['Size_value'].astype(int)

df.drop(columns=['Size_value', 'Size_unit'], inplace=True)
print(df[['Size']].head(10))

### 9. Clean and Convert the Price Column to Numeric

Values of the Price column are strings representing price with special symbol '$'.

In [ ]:

df['Price'] = df['Price'].astype(str)
df['Price'] = df['Price'].str.replace('$', '', regex=False)

df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

df['Price'].fillna(0, inplace=True)
print(df[['Price']].head(20))

### 10. Paid or Free?

Now that you have cleaned the Price column, let's create another auxiliary Distribution column.

This column should contain Free/Paid values depending on the app's price.

In [ ]:

df['Distribution'] = df['Price'].apply(lambda x: 'Paid' if x > 0 else 'Free')
print(df[['Price', 'Distribution']].head(10))


In [ ]:
print(df['Distribution'].value_counts())

## Finally!!! 
- Now all is left is to save the new dataframe we made into a new csv file called `filteredplaystore.csv`

In [ ]:
df.to_csv('filteredplaystore.csv', index=False)


In [ ]:
df_check = pd.read_csv('filteredplaystore.csv')
print(df_check.head())